Librerías

In [3]:
import pandas as pd
import yfinance as yf
from tqdm import tqdm
import requests
from io import StringIO

80 Empresas de mayor valoración de mercado

In [4]:
def get_top_80_sp500_by_market_cap():
    # 1. Obtener la lista de empresas del S&P 500 desde Wikipedia
    print("Obteniendo la lista de empresas del S&P 500...")
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    
    # --- SOLUCIÓN AL ERROR 403 ---
    # Enviamos un 'User-Agent' haciéndonos pasar por un navegador web
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    # Hacemos la petición con requests
    response = requests.get(url, headers=headers)
    
    # Usamos StringIO para que pandas lea el texto HTML sin generar advertencias
    table = pd.read_html(StringIO(response.text))
    df = table[0]
    # -----------------------------
    
    # Extraer los símbolos y los nombres de las empresas
    companies = dict(zip(df['Symbol'], df['Security']))
    tickers = list(companies.keys())

    # Yahoo Finance usa guiones en lugar de puntos para ciertas acciones
    tickers = [ticker.replace('.', '-') for ticker in tickers]

    # 2. Obtener la capitalización de mercado para cada empresa
    data = []
    print("Descargando capitalización de mercado (esto tomará unos minutos)...")
    
    for ticker in tqdm(tickers):
        try:
            stock = yf.Ticker(ticker)
            mcap = stock.info.get('marketCap')
            
            if mcap:
                original_ticker = ticker.replace('-', '.')
                data.append({
                    'Ticker': original_ticker,
                    'Empresa': companies.get(original_ticker, ticker),
                    'Market Cap (USD)': mcap
                })
        except Exception as e:
            continue

    # 3. Crear un DataFrame con los datos recolectados
    mcap_df = pd.DataFrame(data)

    # Ordenar de mayor a menor por Market Cap y reiniciar el índice
    mcap_df = mcap_df.sort_values(by='Market Cap (USD)', ascending=False).reset_index(drop=True)

    # 4. Seleccionar las 80 empresas más grandes
    top_80 = mcap_df.head(80)

    # Formatear la columna de Market Cap
    top_80.loc[:, 'Market Cap (Billions USD)'] = (top_80['Market Cap (USD)'] / 1e9).apply(lambda x: f"${x:,.2f}B")
    top_80 = top_80.drop(columns=['Market Cap (USD)'])

    return top_80

if __name__ == "__main__":
    top_80_companies = get_top_80_sp500_by_market_cap()
    
    print("\n--- Top 80 Empresas del S&P 500 por Valor de Mercado ---")
    pd.set_option('display.max_rows', 80)
    print(top_80_companies)

Obteniendo la lista de empresas del S&P 500...
Descargando capitalización de mercado (esto tomará unos minutos)...


100%|██████████| 503/503 [03:56<00:00,  2.13it/s]


--- Top 80 Empresas del S&P 500 por Valor de Mercado ---
   Ticker                      Empresa Market Cap (Billions USD)
0    NVDA                       Nvidia                $5,316.75B
1   GOOGL      Alphabet Inc. (Class A)                $4,696.67B
2    GOOG      Alphabet Inc. (Class C)                $4,645.91B
3    AAPL                   Apple Inc.                $4,479.50B
4    MSFT                    Microsoft                $3,113.18B
5    AMZN                       Amazon                $2,887.85B
6    AVGO                     Broadcom                $1,962.85B
7    TSLA                  Tesla, Inc.                $1,569.33B
8    META               Meta Platforms                $1,541.79B
9   BRK.B           Berkshire Hathaway                $1,035.25B
10    WMT                      Walmart                  $967.20B
11    LLY                  Lilly (Eli)                  $928.88B
12     MU            Micron Technology                  $859.45B
13    JPM               JPMorgan


C:\Users\cm180\AppData\Local\Temp\ipykernel_30012\608542726.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_80.loc[:, 'Market Cap (Billions USD)'] = (top_80['Market Cap (USD)'] / 1e9).apply(lambda x: f"${x:,.2f}B")


Acciones en los últimos 5 años

In [5]:
def get_top_80_tickers():
    """Obtiene los tickers de las 80 empresas más grandes del S&P 500"""
    print("1. Obteniendo la lista de empresas del S&P 500 desde Wikipedia...")
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    table = pd.read_html(StringIO(response.text))
    df = table[0]
    
    # Reemplazar puntos por guiones para el formato de Yahoo Finance (ej. BRK.B -> BRK-B)
    tickers = [ticker.replace('.', '-') for ticker in df['Symbol']]

    print("2. Descargando capitalización de mercado para identificar el Top 80...")
    mcap_data = []
    for ticker in tqdm(tickers):
        try:
            stock = yf.Ticker(ticker)
            mcap = stock.info.get('marketCap')
            if mcap:
                mcap_data.append({'Ticker': ticker, 'Market Cap': mcap})
        except Exception:
            continue

    # Ordenar y filtrar las 80 principales
    mcap_df = pd.DataFrame(mcap_data)
    mcap_df = mcap_df.sort_values(by='Market Cap', ascending=False).reset_index(drop=True)
    top_80_tickers = mcap_df.head(80)['Ticker'].tolist()
    
    return top_80_tickers

def get_historical_prices(tickers_list):
    """Descarga los precios de cierre de los últimos 5 años para una lista de tickers"""
    print(f"\n3. Descargando precios históricos de los últimos 5 años para las {len(tickers_list)} empresas...")
    
    # yf.download descarga todo de golpe de manera masiva
    historical_data = yf.download(tickers_list, period="5y")
    
    # --- SOLUCIÓN AL KEYERROR ---
    # Revisamos el primer nivel de las columnas (MultiIndex) para ver qué nombres nos devolvió yfinance
    columnas_disponibles = historical_data.columns.get_level_values(0).unique()
    
    if 'Adj Close' in columnas_disponibles:
        prices_df = historical_data['Adj Close']
    elif 'Close' in columnas_disponibles:
        prices_df = historical_data['Close']
    else:
        # En caso de una estructura inesperada, intentamos tomar la primera columna de precios
        print("Advertencia: No se encontró 'Adj Close' ni 'Close'. Usando la estructura alternativa.")
        prices_df = historical_data.iloc[:, :len(tickers_list)]
        
    return prices_df

if __name__ == "__main__":
    # Paso 1 y 2: Identificar el top 80
    top_tickers = get_top_80_tickers()
    
    # Paso 3: Descargar el histórico de 5 años
    df_precios_5_anos = get_historical_prices(top_tickers)
    
    # --- Mostrar resultados ---
    print("\n--- DATAFRAME DE PRECIOS HISTÓRICOS (Últimos 5 años) ---")
    print(f"Dimensiones del DataFrame (Filas/Días x Columnas/Empresas): {df_precios_5_anos.shape}")
    
    # Configurar pandas para ver más columnas en la consola
    pd.set_option('display.max_columns', 10)
    
    print("\nPrimeras 5 filas (Inicio de hace 5 años):")
    print(df_precios_5_anos.head())
    
    print("\nÚltimas 5 filas (Precios actuales):")
    print(df_precios_5_anos.tail())

1. Obteniendo la lista de empresas del S&P 500 desde Wikipedia...
2. Descargando capitalización de mercado para identificar el Top 80...


100%|██████████| 503/503 [03:06<00:00,  2.70it/s]



3. Descargando precios históricos de los últimos 5 años para las 80 empresas...


[*********************100%***********************]  80 of 80 completed


--- DATAFRAME DE PRECIOS HISTÓRICOS (Últimos 5 años) ---
Dimensiones del DataFrame (Filas/Días x Columnas/Empresas): (1255, 80)

Primeras 5 filas (Inicio de hace 5 años):
Ticker            AAPL       ABBV         ABT         ADI        AMAT  ...  \
Date                                                                   ...   
2021-05-24  123.890953  96.092003  106.773514  149.090836  128.963638  ...   
2021-05-25  123.696014  94.972824  107.784004  150.200943  131.761795  ...   
2021-05-26  123.647278  95.088882  106.281952  150.347687  131.416199  ...   
2021-05-27  122.116928  93.115814  105.316978  150.366165  132.299347  ...   
2021-05-28  121.463837  93.845345  106.190933  151.646805  132.596954  ...   

Ticker            WDC       WELL        WFC        WMT        XOM  
Date                                                               
2021-05-24  55.625347  66.005936  40.994263  44.331654  49.599537  
2021-05-25  54.978111  66.156235  40.861282  44.513031  48.476242  
2021-05-2

Análisis exploratorio

### Principales Indicadores financieras de acciones bursatiles
### 1. Precio de Cierre

El precio de cierre no se "calcula" per se, ya que es el último precio registrado de una acción en el mercado al final del día de negociación. Sin embargo, matemáticamente se denota simplemente como el precio en el instante de tiempo $t$:

$$P_t$$

Donde:

* $P_t$: Precio de cierre del activo en el día $t$.
* $P_{t-1}$: Precio de cierre del activo en el día anterior.

### 2. Retornos Diarios

El retorno diario mide la ganancia o pérdida porcentual de un activo de un día para otro. Puede calcularse como un retorno simple o como un retorno logarítmico (muy común en finanzas cuantitativas).

**Retorno Simple:**


$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**Retorno Logarítmico continuo:**


$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

### 3. Volatilidad

La volatilidad histórica de un activo se calcula como la desviación estándar de sus retornos diarios a lo largo de un período de tiempo ($N$).

**Volatilidad diaria (Desviación estándar muestral):**


$$\sigma = \sqrt{\frac{1}{N-1} \sum_{t=1}^{N} (R_t - \bar{R})^2}$$

Donde:

* $R_t$: Retorno del día $t$.
* $\bar{R}$: Retorno promedio en el período $N$.
* $N$: Número de días de observación.

Para anualizar la volatilidad (asumiendo 252 días hábiles de bolsa en un año):


$$\sigma_{anual} = \sigma_{diaria} \times \sqrt{252}$$

### 4. Sharpe Ratio

El Ratio de Sharpe mide el rendimiento excedente por cada unidad de riesgo (volatilidad) asumida en una inversión.

$$S = \frac{R_p - R_f}{\sigma_p}$$

Donde:

* $R_p$: Rendimiento esperado o histórico del portafolio o activo.
* $R_f$: Tasa libre de riesgo (por ejemplo, el rendimiento de los bonos del Tesoro).
* $\sigma_p$: Volatilidad (desviación estándar) del portafolio o activo.

### 5. Drawdown

El drawdown mide la caída porcentual desde el punto máximo histórico (pico) de un activo hasta el valor en el tiempo $t$.

$$DD_t = \frac{P_t - P_{max}}{P_{max}}$$

Para definirlo de forma más rigurosa, si buscamos el drawdown en el momento $t$ comparado con el máximo alcanzado hasta ese momento:


$$DD_t = \frac{P_t - \max_{0 \le \tau \le t} (P_\tau)}{\max_{0 \le \tau \le t} (P_\tau)}$$

### 6. Value at Risk (VaR)

El VaR determina la pérdida máxima esperada durante un período de tiempo específico, dado un nivel de confianza determinado ($1 - \alpha$).

En finanzas, hay dos formas principales de representarlo matemáticamente: el **VaR Histórico** y el **VaR Paramétrico**.

**A. VaR Histórico (Empírico)**
Se define como el cuantil de la distribución empírica de los retornos. Matemáticamente, es el valor $VaR$ tal que la probabilidad de que el retorno $R$ sea menor que ese valor es igual a $\alpha$ (nivel de significancia, típicamente 5% o 1%).

$$P(R < VaR_{\alpha}) = \alpha$$

Alternativamente, expresado como función de cuantil:


$$VaR_{\alpha} = \inf \{ r \in \mathbb{R} : F_R(r) \ge \alpha \}$$

Donde:

* $\alpha$: Nivel de significancia (por ejemplo, 0.05 para un 95% de confianza).
* $F_R(r)$: Función de distribución acumulada empírica de los retornos.
* $R$: Retornos del activo o portafolio.

**B. VaR Paramétrico (Asumiendo Distribución Normal)**
Si asumimos que los retornos se distribuyen normalmente, el VaR se calcula usando la media, la desviación estándar y el valor Z de la distribución normal estándar.

$$VaR_{\alpha} = \mu + Z_{\alpha} \cdot \sigma$$

Donde:

* $\mu$: Retorno esperado o promedio del activo.
* $\sigma$: Volatilidad (desviación estándar) de los retornos.
* $Z_{\alpha}$: Valor crítico de la distribución normal estándar para el nivel de significancia $\alpha$ (ej. -1.645 para un 95% de confianza).


### 7. Beta ($\beta$)

El coeficiente Beta mide la sensibilidad o el riesgo sistemático de un activo individual ($i$) en comparación con el mercado en general ($m$). Es la pendiente de la regresión lineal entre los retornos del activo y los retornos del mercado.

$$\beta_i = \frac{\text{Cov}(R_i, R_m)}{\text{Var}(R_m)}$$

También se puede expresar utilizando las desviaciones estándar y el coeficiente de correlación de Pearson ($\rho$):

$$\beta_i = \rho_{i,m} \frac{\sigma_i}{\sigma_m}$$

Donde:

* $\beta_i$: Beta del activo $i$.
* $\text{Cov}(R_i, R_m)$: Covarianza entre los retornos del activo $i$ y los retornos del mercado $m$ (ej. S&P 500).
* $\text{Var}(R_m)$: Varianza de los retornos del mercado $m$ (que equivale a $\sigma_m^2$).
* $\rho_{i,m}$: Correlación entre los retornos del activo y del mercado.
* $\sigma_i$: Desviación estándar (volatilidad) de los retornos del activo $i$.
* $\sigma_m$: Desviación estándar (volatilidad) de los retornos del mercado $m$.

### Creación de indicadores

In [8]:

def calcular_indicadores(df_precios, risk_free_rate=0.04):
    print("\nCalculando indicadores financieros (incluyendo VaR y Beta)...")
    
    # 1. Retornos Diarios
    # pct_change() calcula (Precio Hoy - Precio Ayer) / Precio Ayer
    retornos_diarios = df_precios.pct_change().dropna()
    
    # Calculamos el rendimiento promedio anualizado (asumiendo 252 días bursátiles)
    rendimiento_anualizado = retornos_diarios.mean() * 252
    
    # 2. Volatilidad Anualizada
    # Desviación estándar de los retornos diarios multiplicada por la raíz de 252
    volatilidad_anualizada = retornos_diarios.std() * np.sqrt(252)
    
    # 3. Sharpe Ratio
    # (Rendimiento del activo - Tasa libre de riesgo) / Volatilidad
    sharpe_ratio = (rendimiento_anualizado - risk_free_rate) / volatilidad_anualizada
    
    # 4. Maximum Drawdown (Máxima caída desde el pico)
    # cummax() calcula el precio máximo histórico hasta la fecha actual
    precios_maximos_historicos = df_precios.cummax()
    drawdowns_diarios = (df_precios - precios_maximos_historicos) / precios_maximos_historicos
    max_drawdown = drawdowns_diarios.min()
    
    # 5. Value at Risk (VaR) Histórico al 95% de confianza
    # Cuantil 5%: nos da el límite de pérdida en el peor 5% de los días
    var_95 = retornos_diarios.quantile(0.05)
    
    # 6. Beta (Sensibilidad respecto al mercado)
    print("Descargando el índice S&P 500 (^GSPC) como benchmark para calcular el Beta...")
    
    # Tomamos las fechas exactas que ya tenemos en nuestro DataFrame
    fecha_inicio = df_precios.index.min()
    fecha_fin = df_precios.index.max() + pd.Timedelta(days=1)
    
    # Descargamos el benchmark
    benchmark_data = yf.download('^GSPC', start=fecha_inicio, end=fecha_fin, progress=False)
    
    # Manejar estructura de columnas de yfinance para el benchmark
    columnas_bench = benchmark_data.columns.get_level_values(0).unique()
    if 'Adj Close' in columnas_bench:
        benchmark_precios = benchmark_data['Adj Close']
    elif 'Close' in columnas_bench:
        benchmark_precios = benchmark_data['Close']
    else:
        benchmark_precios = benchmark_data.iloc[:, 0]
        
    benchmark_retornos = benchmark_precios.pct_change().dropna()
    
    # Si devuelve un DataFrame en lugar de una Serie, tomamos la primera columna
    if isinstance(benchmark_retornos, pd.DataFrame):
        benchmark_retornos = benchmark_retornos.iloc[:, 0]
        
    # Alineamos las fechas de las empresas con las del benchmark (por si hay diferencias de feriados)
    datos_alineados = retornos_diarios.join(benchmark_retornos.rename('Benchmark'), how='inner')
    benchmark_alineado = datos_alineados['Benchmark']
    activos_alineados = datos_alineados.drop(columns=['Benchmark'])
    
    # Cálculo matemático de Beta: Covarianza(activo, mercado) / Varianza(mercado)
    varianza_mercado = benchmark_alineado.var()
    betas = activos_alineados.apply(lambda col: col.cov(benchmark_alineado) / varianza_mercado)
    
    # Consolidar todo en un nuevo DataFrame
    df_indicadores = pd.DataFrame({
        'Rendimiento Anualizado': rendimiento_anualizado,
        'Volatilidad Anualizada': volatilidad_anualizada,
        'Sharpe Ratio': sharpe_ratio,
        'Max Drawdown': max_drawdown,
        'VaR 95% Diario': var_95,
        'Beta': betas
    })
    
    # Ordenar por Sharpe Ratio (de mejor a peor)
    df_indicadores = df_indicadores.sort_values(by='Sharpe Ratio', ascending=False)
    
    return df_indicadores, retornos_diarios, drawdowns_diarios

# --- EJECUCIÓN ---
# Asegúrate de haber ejecutado antes el código que obtiene 'df_precios_5_anos'

df_resumen, df_retornos, df_drawdowns = calcular_indicadores(df_precios_5_anos)

# Formatear la tabla para mostrar porcentajes y hacerla más legible
df_resumen_formateado = df_resumen.copy()
df_resumen_formateado['Rendimiento Anualizado'] = (df_resumen_formateado['Rendimiento Anualizado'] * 100).map("{:.2f}%".format)
df_resumen_formateado['Volatilidad Anualizada'] = (df_resumen_formateado['Volatilidad Anualizada'] * 100).map("{:.2f}%".format)
df_resumen_formateado['Sharpe Ratio'] = df_resumen_formateado['Sharpe Ratio'].map("{:.2f}".format)
df_resumen_formateado['Max Drawdown'] = (df_resumen_formateado['Max Drawdown'] * 100).map("{:.2f}%".format)

# Formatear los nuevos indicadores
df_resumen_formateado['VaR 95% Diario'] = (df_resumen_formateado['VaR 95% Diario'] * 100).map("{:.2f}%".format)
df_resumen_formateado['Beta'] = df_resumen_formateado['Beta'].map("{:.2f}".format)

# Configuramos pandas para mostrar la tabla de forma amplia
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("\n--- RESUMEN DE INDICADORES (Top 80 S&P 500) ---")
print("Ordenado por mejor Sharpe Ratio")
print("-" * 100)
print(df_resumen_formateado.head(80))


Calculando indicadores financieros (incluyendo VaR y Beta)...
Descargando el índice S&P 500 (^GSPC) como benchmark para calcular el Beta...

--- RESUMEN DE INDICADORES (Top 80 S&P 500) ---
Ordenado por mejor Sharpe Ratio
----------------------------------------------------------------------------------------------------
      Rendimiento Anualizado Volatilidad Anualizada Sharpe Ratio Max Drawdown VaR 95% Diario  Beta
SNDK                 348.53%                100.05%         3.44      -47.50%         -6.81%  2.62
WDC                  199.97%                 64.09%         3.06      -60.85%         -5.42%  2.00
STX                  185.97%                 62.43%         2.91      -56.99%         -5.83%  1.69
MU                   187.63%                 67.14%         2.73      -57.63%         -5.63%  2.36
GLW                  117.68%                 50.42%         2.25      -34.96%         -4.53%  1.55
CAT                   78.26%                 34.83%         2.13      -34.05%      